# CSW% 예측 — 투구 단위 xCSW 분해 모델 (설계 + 실행)

*작성 2026-07-23 · 참고자료 검토 · BaseballIQ 누수 분석 · 기존 파이프라인 연계 · 실행 골격*

---

### TL;DR
1. **BaseballIQ는 누수다.** `models/train.py`는 *현재 경기 피처 → 현재 경기 CSW* (Case 1). 목표 `csw_rate`가 피처와 같은 집계에서 나오고, `whiff_rate_delta`는 목표의 분자를 그대로 포함한다. 15개 피처 중 누수 없는 건 `rolling_30d_*` 3개뿐. 구조만 참고하고 피처/시점 설계는 신뢰하지 않는다.
2. **xCSW는 기존 pre-pitch 모델과 질문이 정반대.** "실제로 던져진 공"이 주어졌을 때의 CSW라서, 현재 투구의 **물리·위치·구종이 필수 입력**이고 금지 대상은 **결과(description·events·타구질·WPA)뿐**이다.
3. **분해:** `P(CSW) = P(swing)·P(whiff|swing) + P(take)·P(called|take)` → 투수·구종 단위 **xCSW**로 집계. 이 노트북은 합성 데이터로 end-to-end 실행되며, 로컬에서 실데이터 parquet로 그대로 전환된다.

### 목차

**Part I — 배경과 분석 (읽기)**
&nbsp;&nbsp;1. 두 모델의 관계 *(가장 중요)*
&nbsp;&nbsp;2. 참고자료 6종 검토 요약
&nbsp;&nbsp;3. BaseballIQ 데이터 누수 분석

**Part II — xCSW 설계 (읽기)**
&nbsp;&nbsp;4. 분해 공식 · 3개 모델 · 라벨 정의
&nbsp;&nbsp;5. xCSW 전용 누수 규칙 (기존과 반대)
&nbsp;&nbsp;6. 검증 전략 · Baseline

**Part III — 실행 (코드)**
&nbsp;&nbsp;7. 설정 · 라벨/누수 집합
&nbsp;&nbsp;8. 데이터 로드 (실데이터/합성)
&nbsp;&nbsp;9. 라벨 3종 + 정합성 검증
&nbsp;&nbsp;10. 피처셋 + 결과-누수 안전망
&nbsp;&nbsp;11. 모델 A·B·C 학습 + 결합
&nbsp;&nbsp;12. 투구 단위 평가 + Calibration/ECE
&nbsp;&nbsp;13. 투수·구종 xCSW 집계
&nbsp;&nbsp;14. Baseline 대비

**Part IV — 다음 단계**

---
# Part I — 배경과 분석

## 1. 두 모델의 관계 *(가장 중요)*

팀은 이미 **엄격한 투구 전(strict pre-pitch) `is_csw` 모델**을 완성했다(`train_csw_model.py`). 이 노트북의 xCSW는 그것을 **대체가 아니라 보완**한다. 두 모델은 질문이 정반대이고 **따라서 누수 규칙도 정반대**다.

| 구분 | 기존: pre-pitch `is_csw` | 신규: 투구 단위 **xCSW** (이 노트북) |
|---|---|---|
| 질문 | 공을 **던지기 전** CSW 확률 | **던져진 공**(구속·무브·위치)의 CSW 확률 |
| 성격 | 예측 · 시퀀싱/상황/커맨드 | 기술·기대값 · "stuff + location" 품질 |
| 현재 물리(구속·무브·회전·릴리스) | ❌ 금지 | ✅ **필수 입력** |
| 현재 위치(`plate_x/z`,`zone`,`sz_*`) | ❌ 금지 | ✅ **필수 입력** (C의 핵심) |
| 현재 구종 `pitch_type` | ❌ 금지 | ✅ 입력/층화 |
| 금지 대상 | 물리·위치·구종 + 결과 전부 | **결과만** (description·events·타구질·WPA) |
| 검증 | 연도 분할 2017–18 / 2019 | 동일 |

**연결 고리 3가지**: ① 투수·구종 xCSW로 *운·맥락 보정된 CSW 실력* 평가 · ② 과거 xCSW를 **집계·shift**하여 pre-pitch 모델 피처로 투입 · ③ 다음 경기 CSW(1안)를 구종 믹스로 조립하는 상향식 경로.

## 2. 참고자료 6종 검토 요약

| # | 자료 | 핵심 기여 | 이 프로젝트 반영 | 주의 |
|---|---|---|---|---|
| 1 | **BaseballIQ** | Statcast→DuckDB→XGBoost CSW→SHAP→Streamlit 구조 | 폴더·수집·집계·대시보드 **구조만** | **목표 누수** (3장) → 반면교사 |
| 2 | **PyMC-BART** (whiff) | 스윙→헛스윙 확률 BART. 구속·수직/수평무브·회전(+릴리스·axis·platoon·구장). 상관≈0.85 | **모델 B** 피처·calibration 근거 | whiff 성분만. 연도 holdout 재평가 |
| 3 | **CalledStrike** (R GAM) | taken만 `logit P=s(plate_x,plate_z)`, 존 확률표면 | **모델 C** 뼈대 (Py는 pygam/GBM/BART) | 카운트·좌우·구종·`sz_*`·주심 확장 |
| 4 | **Plate-discipline BART** (arXiv 2305.05752) | 3단계 분해: called\|take · contact\|swing · 기대득점 | **분해 구조 이론 근거** | 목적은 스윙 의사결정 평가 |
| 5 | **Baseball Scouting Lab** | Stuff/Location/Called-strike 분리모델, AUC+Brier 병기 | 모델 분리·평가지표 | 일부 **랜덤 분할** → 쓰지 말 것 |
| 6 | **pybaseball** | Savant 투구단위 수집 표준 | 이미 2017–19 2.2M행 수집 완료 | 403/제한 → 월별분할·캐싱으로 해결됨 |

모델 B←(2), 모델 C←(3)(5), 분해 골격←(4).

## 3. BaseballIQ 데이터 누수 분석 (코드 직접 검토)

### 판정 — **Case 1: 현재 경기 피처 → 현재 경기 CSW (심각한 목표 누수)**
`models/train.py`에 목표를 다음 경기로 옮기는 `shift(-1)`이 **없다**. README의 "Next start projected CSW"는 코드와 **불일치**(실제 목표는 같은 행 `csw_rate`).

```python
# models/train.py
TARGET = "csw_rate"                         # pitcher_game_summary의 "그 경기" CSW
FEATURE_COLS = [ ... "whiff_rate_delta", "zone_rate", "chase_rate",
                 "barrel_rate_allowed", "avg_xwoba_allowed", "stuff_diversity",
                 "velo_vs_30d_avg", ... "total_pitches" ]
```
```sql
-- feature_engineering.py: 아래가 모두 같은 game_agg (동일 투수-경기)에서 계산됨
whiff_rate = SUM(swinging_strike)/SUM(swing)
csw_rate   = SUM(called_strike + swinging_strike)/COUNT(*)   -- ← 목표
-- whiff_rate_delta = 현재 whiff_rate − 30일평균  → 목표 분자(swinging_strike)를 그대로 포함
```

### 피처별 누수 판정 (15개 중 누수-없음 3개뿐)

| 피처 | 시점 | 누수 |
|---|---|---|
| `rolling_30d_avg_velo` / `rolling_30d_whiff_rate` / `rolling_30d_csw_rate` | 과거 30일 (현재 제외) | ✅ 안전 |
| `whiff_rate_delta` | 현재−과거, **현재 whiff 포함** | ❌ 심각 |
| `velo_vs_30d_avg` | 현재−과거 | ❌ |
| `zone_rate`,`chase_rate` | 현재 경기 | ❌ |
| `barrel_rate_allowed`,`avg_xwoba_allowed` | 현재 경기 결과 | ❌ |
| `stuff_diversity`,`avg_spin`,`avg_h_break`,`avg_v_break` | 현재 경기 | ❌ |
| `total_pitches` | 경기 종료 후 확정 | ❌ |
| `home_away` | `np.random.randint` 임시값 | ⚠️ 노이즈 |

### TimeSeriesSplit이 못 막는 이유
행(경기) **사이**의 시간 순서만 지킬 뿐, 한 행 **내부**의 피처↔목표 누수(같은 경기 값으로 그 경기 목표 예측)는 전혀 막지 못한다. 우리 팀은 이미 `CURRENT_PITCH_LEAKAGE` + `integrity_report` assert로 이 부류를 차단 중 — 방향이 정확하다. xCSW는 물리·위치를 *의도적으로* 쓰므로 같은 안전망을 **결과 열 전용**으로 다시 정의한다(5장).

---
# Part II — xCSW 설계

## 4. 분해 공식 · 3개 모델 · 라벨 정의

```text
P(CSW_i) = P(swing_i)·P(whiff_i|swing_i) + (1−P(swing_i))·P(called_i|take_i)
         = p_swing · p_whiff            + (1 − p_swing) · p_cs_take
```
투수·경기·구종 단위로 `p_csw_i` 평균 → **xCSW**.

| 모델 | 대상 투구 | 라벨(양성) | 핵심 피처 | 선행연구 | 알고리즘 |
|---|---|---|---|---|---|
| **A. P(swing)** | 전체 | `is_swing` | 위치·카운트·좌우·구종·(맥락)물리 | plate-discipline BART | HGB |
| **B. P(whiff\|swing)** | 스윙만 | `is_whiff` | 구속·수직/수평무브·회전·릴리스·VAA/HAA·platoon·구장 | PyMC-BART | HGB / **BART** |
| **C. P(called\|take)** | 테이크만 | `is_called_strike` | **`plate_x`,`plate_z`**·`sz_*`·카운트·좌우·구종·주심 | CalledStrike GAM | HGB / pygam / BART |

**라벨(Statcast `description`)** — 팀 `is_csw`와 정합:
```python
WHIFF  = {"swinging_strike","swinging_strike_blocked"}
CALLED = {"called_strike"}
SWING  = {"swinging_strike","swinging_strike_blocked","foul","foul_tip",
          "hit_into_play","foul_bunt","missed_bunt","bunt_foul_tip"}
```
**정합성**: `is_csw == (swing & whiff) | (~swing & called)` 가 성립해야 함. 유일한 경계는 `missed_bunt`(whiff이나 CSW 아님) — 9장에서 assert로 감지.

## 5. xCSW 전용 누수 규칙 (기존과 반대)

| 범주 | 예시 열 | pre-pitch | **xCSW** |
|---|---|---|---|
| 투구 물리 | `release_speed`,`release_spin_rate`,`pfx_x/z`,`vx0..az`,`release_pos_*`,`release_extension`,`spin_axis` | ❌ | ✅ 입력 |
| 투구 위치 | `plate_x`,`plate_z`,`zone`,`sz_top`,`sz_bot` | ❌ | ✅ 입력 |
| 현재 구종 | `pitch_type`,`pitch_name` | ❌ | ✅ |
| 상황 | 카운트·주자·점수차·좌우·매치업 | ✅ | ✅ |
| **결과(양쪽 금지)** | `description`,`type`,`events`,`launch_*`,`estimated_woba*`,`delta_run_exp`,`delta_home_win_exp`,post-score | ❌ | ❌ **라벨용, 피처 금지** |

**왜 반대인가**: pre-pitch 누수 = "릴리스 후에만 아는 값 전부". xCSW는 바로 그 값들이 입력(=stuff+location의 정의). 따라서 xCSW 누수는 오직 **투구 결과**. `zone`은 `plate_x/z`와 중복이니 후자 사용 시 제외.

## 6. 검증 전략 · Baseline

**분할(랜덤 금지)**: Train 2017–18 / Test 2019 → (추가 수집 시) Val 2024·Test 2025·실시간 2026. 집단통계(주심·포수 등)도 train 연도로만 산출 후 조인.

**투구 단위 지표**: Log loss(주), Brier, ROC-AUC / PR-AUC(희소 B·C는 PR 중시), Calibration curve, ECE.
**집계 지표(경기·투수·구종)**: MAE·RMSE·R²·실제 CSW 상관·투수별/구종별 calibration.

**Baseline(필수)**: ① 리그평균 CSW ② 투수 직전30일 CSW ③ 투수 전년도 CSW ④ 투수별 shrinkage 평균. + 투구단위엔 *위치만 로지스틱*, *구속·무브만 모델*도 참고선. **분해 모델이 이들을 확실히 이기지 못하면 예측력을 주장하지 않는다.**

---
# Part III — 실행

## 7. 설정 · 라벨/누수 집합

In [ ]:
from __future__ import annotations
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd

RANDOM_STATE = 0
rng = np.random.default_rng(RANDOM_STATE)

# 실데이터 경로(팀 기존 산출물). 없으면 자동 합성.
RAW_CSW_FILE = Path("data/statcast_2017_2019_raw_csw.parquet")
USE_SYNTHETIC = not RAW_CSW_FILE.exists()

TRAIN_YEARS = (2017, 2018); TEST_YEARS = (2019,)
SEQUENCE_COLUMNS = ["game_date", "game_pk", "at_bat_number", "pitch_number"]

# 라벨 정의 (Statcast description)
WHIFF  = {"swinging_strike", "swinging_strike_blocked"}
CALLED = {"called_strike"}
SWING  = {"swinging_strike", "swinging_strike_blocked", "foul", "foul_tip",
          "hit_into_play", "foul_bunt", "missed_bunt", "bunt_foul_tip"}

# xCSW 누수 안전망: 결과 열은 라벨 산출에만, 피처에서 제외
XCSW_OUTCOME_LEAKAGE = {
    "description", "type", "events", "bb_type",
    "launch_speed", "launch_angle", "hit_distance_sc",
    "estimated_ba_using_speedangle", "estimated_woba_using_speedangle",
    "woba_value", "babip_value", "iso_value",
    "delta_run_exp", "delta_home_win_exp", "home_win_exp", "bat_win_exp",
    "post_home_score", "post_away_score", "post_bat_score", "post_fld_score",
    "is_csw", "is_swing", "is_whiff", "is_called_strike",
}
print("USE_SYNTHETIC =", USE_SYNTHETIC)

## 8. 데이터 로드 (실데이터 / 합성)

실데이터가 있으면 필요한 열만 읽고, 없으면 **물리/위치→결과 관계가 살아있는 합성 Statcast**를 생성한다(모델이 실제 신호를 학습하는지 스모크 테스트 가능).

In [ ]:
PITCH_PARAMS = {  # (velo, pfx_x, pfx_z, spin) 대략 평균
    "FF": (94, 0.4, 1.4, 2300), "SI": (93, 1.4, 0.8, 2150),
    "FC": (89, -0.2, 0.6, 2400), "SL": (85, -0.5, 0.2, 2450),
    "CU": (79, -0.7, -1.2, 2600), "CH": (85, 1.1, 0.5, 1750),
}
PITCH_MIX = np.array([0.38, 0.15, 0.09, 0.18, 0.10, 0.10]); PITCH_CODES = list(PITCH_PARAMS)
def _sigmoid(z): return 1.0 / (1.0 + np.exp(-z))

def make_synthetic_statcast(n=120_000, n_pitchers=80, seed=0) -> pd.DataFrame:
    r = np.random.default_rng(seed)
    year = r.choice([2017, 2018, 2019], n, p=[0.33, 0.33, 0.34])
    pt = np.array(PITCH_CODES)[r.choice(len(PITCH_CODES), n, p=PITCH_MIX)]
    base = np.array([PITCH_PARAMS[c] for c in pt])
    velo = base[:, 0] + r.normal(0, 1.4, n); spin = base[:, 3] + r.normal(0, 150, n)
    pfx_x = base[:, 1] + r.normal(0, 0.25, n); pfx_z = base[:, 2] + r.normal(0, 0.25, n)
    sz_top = r.normal(3.4, 0.15, n); sz_bot = r.normal(1.6, 0.12, n)
    plate_x = r.normal(0.0, 0.75, n); plate_z = r.normal((sz_top + sz_bot) / 2, 0.75, n)
    balls = r.integers(0, 4, n); strikes = r.integers(0, 3, n)
    stand = r.choice(["R", "L"], n, p=[0.56, 0.44]); p_throws = r.choice(["R", "L"], n, p=[0.72, 0.28])
    rel_x = np.where(p_throws == "R", -1.8, 1.8) + r.normal(0, 0.3, n); rel_z = r.normal(5.9, 0.25, n)

    zdist = np.sqrt((plate_x / 0.83) ** 2 + ((plate_z - (sz_top + sz_bot) / 2) / ((sz_top - sz_bot) / 2)) ** 2)
    p_swing = _sigmoid(2.4 * (1.0 - zdist) + 0.35 * strikes - 0.25 * balls)
    swing = r.random(n) < p_swing
    movement = np.abs(pfx_x) + np.abs(pfx_z); breaking = np.isin(pt, ["SL", "CU"]).astype(float)
    p_whiff = _sigmoid(-1.6 + 0.10 * (velo - 88) + 0.5 * movement + 0.6 * (zdist - 1.0) + 0.35 * breaking)
    whiff = swing & (r.random(n) < p_whiff)
    p_called = _sigmoid(5.0 * (1.0 - zdist)); called = (~swing) & (r.random(n) < p_called)

    desc = np.where(whiff, "swinging_strike",
            np.where(swing, np.where(r.random(n) < 0.42, "foul", "hit_into_play"),
            np.where(called, "called_strike", "ball")))
    typ = np.where(np.isin(desc, ["swinging_strike", "called_strike", "foul"]), "S",
           np.where(desc == "hit_into_play", "X", "B"))
    df = pd.DataFrame(dict(
        game_year=year, game_pk=r.integers(0, n // 80 + 1, n),
        at_bat_number=r.integers(1, 80, n), pitch_number=r.integers(1, 8, n),
        pitcher=r.integers(1, n_pitchers + 1, n), batter=r.integers(1, 400, n),
        pitch_type=pt, release_speed=velo, release_spin_rate=spin, pfx_x=pfx_x, pfx_z=pfx_z,
        plate_x=plate_x, plate_z=plate_z, sz_top=sz_top, sz_bot=sz_bot,
        release_pos_x=rel_x, release_pos_z=rel_z, balls=balls, strikes=strikes,
        stand=stand, p_throws=p_throws, description=desc, type=typ))
    df["game_date"] = pd.to_datetime(df["game_year"].astype(str) + "-06-01") + pd.to_timedelta(r.integers(0, 120, n), "D")
    return df

RAW_COLS = ["game_year", "game_date", "game_pk", "at_bat_number", "pitch_number", "pitcher",
            "batter", "pitch_type", "release_speed", "release_spin_rate", "pfx_x", "pfx_z",
            "plate_x", "plate_z", "sz_top", "sz_bot", "release_pos_x", "release_pos_z",
            "vx0", "vy0", "vz0", "ax", "ay", "az", "balls", "strikes", "stand", "p_throws",
            "description", "type"]

def load_pitches() -> pd.DataFrame:
    if USE_SYNTHETIC:
        return make_synthetic_statcast(seed=RANDOM_STATE)
    import pyarrow.parquet as pq
    avail = set(pq.ParquetFile(RAW_CSW_FILE).schema_arrow.names)
    df = pd.read_parquet(RAW_CSW_FILE, columns=[c for c in RAW_COLS if c in avail])
    return (df[df["game_type"].eq("R")] if "game_type" in df else df).reset_index(drop=True)

pitches = load_pitches()
print(f"{len(pitches):,} pitches | years {sorted(pitches['game_year'].unique())}")
pitches.head(3)

## 9. 라벨 3종 + 정합성 검증
`is_csw == (is_swing & is_whiff) | (~is_swing & is_called_strike)` 확인. 경계는 `missed_bunt`뿐.

In [ ]:
def add_labels(df: pd.DataFrame) -> pd.DataFrame:
    d = df["description"].astype("string"); out = df.copy()
    out["is_swing"]         = d.isin(SWING).astype("int8")
    out["is_whiff"]         = d.isin(WHIFF).astype("int8")
    out["is_called_strike"] = d.isin(CALLED).astype("int8")
    out["is_take"]          = (1 - out["is_swing"]).astype("int8")
    out["is_csw"]           = d.isin(CALLED | WHIFF).astype("int8")
    return out

pitches = add_labels(pitches)
reconstructed = ((pitches["is_swing"] & pitches["is_whiff"]) |
                 (~pitches["is_swing"].astype(bool) & pitches["is_called_strike"])).astype("int8")
mismatch = int((reconstructed != pitches["is_csw"]).sum())
bunt_edge = int(pitches["description"].eq("missed_bunt").sum())
print(f"CSW 정합성 불일치: {mismatch}  (missed_bunt 경계행 {bunt_edge})")
print(pitches[["is_swing", "is_whiff", "is_called_strike", "is_csw"]].mean().round(4).to_dict())
assert mismatch <= bunt_edge, "CSW 분해 정합성 실패 — 라벨 집합 재점검"

## 10. 피처셋 + 결과-누수 안전망
물리·위치·구종·상황을 입력으로, **결과 열은 전부 제외**. 실데이터에 운동학이 있으면 `kirby_index.add_release_angles()`로 VRA/HRA 추가.

In [ ]:
CATEGORICAL = ["pitch_type", "stand", "p_throws"]
PHYSICS = ["release_speed", "release_spin_rate", "pfx_x", "pfx_z", "release_pos_x", "release_pos_z"]
LOCATION = ["plate_x", "plate_z", "sz_top", "sz_bot"]; CONTEXT = ["balls", "strikes"]

def maybe_add_release_angles(df: pd.DataFrame) -> list[str]:
    if not {"vx0", "vy0", "vz0", "ax", "ay", "az"}.issubset(df.columns):
        return []
    try:
        from kirby_index import add_release_angles
        ang = add_release_angles(df)
        df["vra_deg"], df["hra_deg"] = ang["vra_deg"].values, ang["hra_deg"].values
        return ["vra_deg", "hra_deg"]
    except Exception as e:
        print("release-angle 생략:", e); return []

angle_feats = maybe_add_release_angles(pitches)
FEATURES = PHYSICS + LOCATION + CONTEXT + angle_feats + CATEGORICAL
leak = XCSW_OUTCOME_LEAKAGE & set(FEATURES)
assert not leak, f"결과 누수 열이 피처에 있음: {leak}"
print(f"피처 {len(FEATURES)}개 | 누수 검사 통과")

def to_matrix(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """범주형 one-hot (버전 무관 이식성). 실전 모듈에선 train_csw_model.py처럼 HGB 네이티브 categorical도 가능."""
    X = df[features].copy(); cats = [c for c in features if c in CATEGORICAL]
    return pd.get_dummies(X, columns=cats, dummy_na=True) if cats else X

train_mask = pitches["game_year"].isin(TRAIN_YEARS).to_numpy()
test_mask  = pitches["game_year"].isin(TEST_YEARS).to_numpy()
print(f"train {train_mask.sum():,} / test {test_mask.sum():,}")

## 11. 모델 A·B·C 학습 + 결합
A=전체(`is_swing`), B=스윙만(`is_whiff`), C=테이크만(`is_called_strike`). 기본 `HistGradientBoostingClassifier`(팀 스택 일관); 정밀 calibration 필요 시 B/C를 PyMC-BART로 교체.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

def train_prob_model(df, mask_subset, label, *, max_iter=200):
    sub = df[train_mask & mask_subset]
    clf = HistGradientBoostingClassifier(max_iter=max_iter, learning_rate=0.08, max_leaf_nodes=63,
        min_samples_leaf=200, l2_regularization=1.0, random_state=RANDOM_STATE)
    clf._cols = to_matrix(sub, FEATURES).columns
    clf.fit(to_matrix(sub, FEATURES), sub[label].astype("int8"))
    return clf

def predict_prob(clf, df):
    return clf.predict_proba(to_matrix(df, FEATURES).reindex(columns=clf._cols, fill_value=0))[:, 1]

swing_mask = pitches["is_swing"].astype(bool).to_numpy(); take_mask = ~swing_mask
model_A = train_prob_model(pitches, np.ones(len(pitches), bool), "is_swing")
model_B = train_prob_model(pitches, swing_mask, "is_whiff")
model_C = train_prob_model(pitches, take_mask,  "is_called_strike")

pitches["p_swing"]   = predict_prob(model_A, pitches)
pitches["p_whiff"]   = predict_prob(model_B, pitches)
pitches["p_cs_take"] = predict_prob(model_C, pitches)
pitches["p_csw"] = pitches["p_swing"] * pitches["p_whiff"] + (1 - pitches["p_swing"]) * pitches["p_cs_take"]
print(pitches.loc[test_mask, ["p_swing", "p_whiff", "p_cs_take", "p_csw"]].mean().round(4).to_dict())
print("실제 test CSW율:", round(float(pitches.loc[test_mask, "is_csw"].mean()), 4))

## 12. 투구 단위 평가 + Calibration/ECE
각 모델은 **자기 대상 투구에서만** 평가(B=스윙, C=테이크). 결합 `p_csw`는 전체에서 `is_csw`로.

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score, log_loss, brier_score_loss

def expected_calibration_error(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1); ece = 0.0
    for b in range(bins):
        m = idx == b
        if m.any(): ece += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(ece)

def report(name, y, p):
    y = np.asarray(y).astype(int); p = np.asarray(p, float)
    return dict(model=name, n=len(y), base_rate=round(y.mean(), 4),
                log_loss=round(log_loss(y, p, labels=[0, 1]), 4), brier=round(brier_score_loss(y, p), 4),
                roc_auc=round(roc_auc_score(y, p), 4) if y.min() != y.max() else float("nan"),
                pr_auc=round(average_precision_score(y, p), 4), ece=round(expected_calibration_error(y, p), 4))

te = pitches[test_mask]
te_swing = te[te["is_swing"].astype(bool)]; te_take = te[~te["is_swing"].astype(bool)]
rows = [report("A: P(swing)", te["is_swing"], te["p_swing"]),
        report("B: P(whiff|swing)", te_swing["is_whiff"], te_swing["p_whiff"]),
        report("C: P(called|take)", te_take["is_called_strike"], te_take["p_cs_take"]),
        report("combined p_csw", te["is_csw"], te["p_csw"])]
pd.DataFrame(rows).set_index("model")

## 13. 투수 · 구종 단위 xCSW 집계 (실제 vs 기대)
`p_csw`를 투수/구종으로 평균 → **xCSW**. 실제 CSW와 MAE·RMSE·R²·상관 비교.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def agg_xcsw(df, keys, min_pitches=50):
    g = df.groupby(keys).agg(actual_csw=("is_csw", "mean"), xcsw=("p_csw", "mean"), n=("is_csw", "size"))
    return g[g["n"] >= min_pitches].reset_index()

by_pitcher = agg_xcsw(te, ["pitcher"]); a, x = by_pitcher["actual_csw"], by_pitcher["xcsw"]
print("투수 단위:", dict(n=len(by_pitcher), mae=round(mean_absolute_error(a, x), 4),
      rmse=round(mean_squared_error(a, x) ** 0.5, 4), r2=round(r2_score(a, x), 4),
      corr=round(float(np.corrcoef(a, x)[0, 1]), 4)))
agg_xcsw(te, ["pitch_type"], min_pitches=200).sort_values("xcsw", ascending=False).round(4)

## 14. Baseline 대비 (필수)
분해 모델이 단순 baseline을 **확실히** 이기지 못하면 예측력을 주장하지 않는다.

In [ ]:
from sklearn.linear_model import LogisticRegression

league = float(pitches.loc[train_mask, "is_csw"].mean())
tr = pitches[train_mask]; pg = tr.groupby("pitcher")["is_csw"].agg(["mean", "size"]); lam = 200.0
pg["shrunk"] = (pg["size"] * pg["mean"] + lam * league) / (pg["size"] + lam)
base_pitcher = te["pitcher"].map(pg["shrunk"].to_dict()).fillna(league).to_numpy()

loc_cols = ["plate_x", "plate_z", "sz_top", "sz_bot"]
lr = LogisticRegression(max_iter=1000).fit(pitches.loc[train_mask, loc_cols].fillna(0), pitches.loc[train_mask, "is_csw"])
base_loc = lr.predict_proba(te[loc_cols].fillna(0))[:, 1]

y = te["is_csw"].to_numpy()
pd.DataFrame([
    report("baseline: league mean", y, np.full(len(y), league)),
    report("baseline: pitcher shrinkage", y, base_pitcher),
    report("baseline: location-only LR", y, base_loc),
    report("xCSW (A·B·C 결합)", y, te["p_csw"].to_numpy()),
]).set_index("model")[["log_loss", "brier", "roc_auc", "pr_auc", "ece"]]

---
# Part IV — 다음 단계

1. 로컬에서 `USE_SYNTHETIC=False` → 실데이터(2.2M투구) 실행 (셀 그대로).
2. 모델 B/C에 **VRA/HRA·axis·platoon·구장/주심** 피처 추가 (참고자료 2·3·5).
3. B/C를 **PyMC-BART**로 교체 → posterior·partial dependence·변수중요도·credible interval.
4. `build_xcsw_dataset.py` / `train_xcsw_model.py`로 모듈화 (`build_statcast_strike_dataset.py` 규약, `reports/xcsw_model/` 저장).
5. 과거 **xCSW 집계값(shift)** 을 pre-pitch/다음경기(1안) 모델 피처로 투입.

> 동반 문서: `xcsw_model_design.md` (동일 내용의 산문형 설계서).